In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 🏠 House Price Prediction using Linear Models
Notebook Version B – Feature Engineering + Ridge Regression
# Author: Anshika Bhardwaj
This notebook includes manual preprocessing, outlier removal and Ridge regression.

# 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score


 # 2. Load Data

In [ ]:
import pandas as pd
data = pd.read_csv("/kaggle/input/housedata/data.csv")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
data = pd.read_csv("/kaggle/input/housedata/data.csv")
data.head()

In [ ]:

plt.figure(figsize=(6,4)); plt.hist(data['price'], bins=30); plt.show()

In [ ]:
plt.figure(figsize=(6,4)); plt.boxplot(data['price']); plt.show()

In [ ]:
if 'GrLivArea' in data.columns:
    plt.figure(figsize=(6,4)); plt.scatter(data['GrLivArea'], data['price']); plt.show()

In [ ]:
missing=data.isnull().sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(7,4)); plt.bar(missing.index,missing.values); plt.xticks(rotation=60); plt.show()

In [ ]:
data=data[data['price']<data['price'].quantile(0.99)]
data['TotalSF']=data.get('TotalBsmtSF',0)+data.get('1stFlrSF',0)+data.get('2ndFlrSF',0)
X=pd.get_dummies(data.drop('price',axis=1))
y=np.log1p(data['price'])

In [ ]:
X_scaled=MinMaxScaler().fit_transform(X)

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X_scaled,y,test_size=0.25,random_state=0)

In [ ]:
model=Ridge(alpha=10)
model.fit(X_train,y_train)
pred=model.predict(X_test)
print(mean_absolute_error(y_test,pred), r2_score(y_test,pred))

 # 3. Remove Outliers

In [ ]:
data = data[data['price'] < data['price'].quantile(0.99)]

# 4. Feature Engineering

In [ ]:
data['TotalSF'] = data.get('TotalBsmtSF',0) + data.get('1stFlrSF',0) + data.get('2ndFlrSF',0)

target = 'price'
X = data.drop(target, axis=1)
y = np.log1p(data[target])  # log transform

 # 5. Encoding & Missing Values

In [ ]:
X = X.fillna(0)
X = pd.get_dummies(X)

 # 6. Scaling

In [ ]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# 7. Split Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=0
)

 # 8. Train Ridge Regression

In [ ]:
model = Ridge(alpha=10)
model.fit(X_train, y_train)

 # 9. Evaluate

In [ ]:
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print('MAE:', mae)
print('R2:', r2)

# 10. Feature Importance (coefficients)

In [ ]:
coef = pd.Series(model.coef_, index=X.columns)
coef.sort_values(ascending=False).head(10)

# 11. Save Predictions

In [ ]:
final_pred = np.expm1(model.predict(X_scaled))
pd.DataFrame({'Prediction': final_pred}).to_csv('submission_B.csv', index=False)